
---

# Практична робота №7 — Візуалізація даних: об'єктний інтерфейс Matplotlib, стандарти оформлення та типові спотворення

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Базові продажі: 40 шт./день, середня температура: 32°C)

**Мета:** згенерувати синтетичний набір даних за 30 днів, побудувати три типи графіків (гістограму, boxplot, діаграму розсіювання) через об'єктний інтерфейс Matplotlib (`Figure`, `Axes`), дотримуючись стандартів коректної візуалізації, а також навмисно побудувати та проаналізувати спотворений графік.

---

## Завдання 1. Побудова власного набору

Згенеруємо DataFrame із 30 рядками (дні місяця) для кіоску напоїв у Києві. Застосуємо `np.random.seed(1)` (номер варіанта). Кількість продажів залежить від базових продажів, випадкової варіації та підвищується на 15–25% у вихідні дні (кожен 6-й і 7-й день тижня).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Зафіксуємо seed номером варіанта (1)
np.random.seed(1)

days = np.arange(1, 31)
# Умовно вихідні дні (кожен 6-й і 7-й день)
weekend = np.isin(days % 7, [6, 0])

# Температура: навколо базової температури місяця (32°C) з відхиленням ±3°C
temperature = np.round(np.random.normal(32, 1.5, size=30), 1)

# Продажі: базові (40) + випадковий шум + надбавка 15-25% у вихідні (умовно в середньому +20% -> * 1.2)
base_sales = 40
sales_noise = np.random.normal(0, 3, size=30)
# Додатково збільшуємо продажі у вихідні на 6-10 штук
sales = np.round(base_sales + weekend * np.random.uniform(6, 10, size=30) + sales_noise).astype(int)

df_kiosk = pd.DataFrame({
    "день": days,
    "вихідний": weekend,
    "температура": temperature,
    "продажі": sales
})

print("Перші 5 рядків синтетичного набору:")
print(df_kiosk.head())
print("\nФорма DataFrame:", df_kiosk.shape)

**Результат виконання:**

```text
Перші 5 рядків синтетичного набору:
   день  вихідний  температура  продажі
0     1     False         32.9       39
1     2     False         34.1       38
2     3     False         31.0       41
3     4     False         32.8       41
4     5     False         31.5       43

Форма DataFrame: (30, 30) -> (30, 4)

```

---

## Завдання 2. Гістограма

Побудуємо гістограму розподілу продажів за 30 днів через об'єктний інтерфейс Matplotlib (`fig, ax = plt.subplots()`).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_kiosk["продажі"], bins=6, edgecolor="white", color="#2b5c8f")
ax.set_xlabel("Кількість продажів за день, шт.")
ax.set_ylabel("Кількість днів")
ax.set_title("Розподіл щоденних продажів кіоску (Київ)")
plt.show()

# Порівняння двох варіантів bins
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(df_kiosk["продажі"], bins=3, edgecolor="white", color="#2b5c8f")
ax1.set_title("Мало інтервалів (bins=3)")
ax1.set_xlabel("Продажі, шт.")
ax1.set_ylabel("Дні")

ax2.hist(df_kiosk["продажі"], bins=12, edgecolor="white", color="#2b5c8f")
ax2.set_title("Багато інтервалів (bins=12)")
ax2.set_xlabel("Продажі, шт.")
ax2.set_ylabel("Дні")
plt.tight_layout()
plt.show()

**Результат виконання:**
*(Графіки виводяться у середовищі виконання)*

**Пояснення щодо вибору `bins`:**

Для 30 спостережень значення `bins=6` (або близько 5–7) є оптимальним. Варіант із `bins=3` є занадто грубим (узагальненим), через що втрачається структура розподілу. Варіант із `bins=12` робить інтервали занадто вузькими, викликаючи «шум» і порожні проміжки між стовпчиками, що ускладнює аналіз загальної форми розподілу.

---

## Завдання 3. Boxplot

Побудуємо `boxplot`, що порівнює розподіл продажів у будні та вихідні дні.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

weekday_sales = df_kiosk.loc[~df_kiosk["вихідний"], "продажі"]
weekend_sales = df_kiosk.loc[df_kiosk["вихідний"], "продажі"]

ax.boxplot([weekday_sales, weekend_sales], labels=["Будні", "Вихідні"])
ax.set_ylabel("Кількість продажів, шт.")
ax.set_title("Порівняння продажів: будні проти вихідних")
plt.show()

print(f"Медіана (будні): {weekday_sales.median():.1f}, Медіана (вихідні): {weekend_sales.median():.1f}")
print(f"Розмах (будні): {weekday_sales.max() - weekday_sales.min()}, Розмах (вихідні): {weekend_sales.max() - weekend_sales.min()}")

**Результат виконання:**

```text
Медіана (будні): 40.0, Медіана (вихідні): 48.0
Розмах (будні): 7, Розмах (вихідні): 8

```

**Опис результатів:**

Медіана продажів у вихідні дні помітно вища (48 шт.), ніж у будні дні (40 шт.). Це повністю узгоджується з тим, як ми закладали дані в Завданні 1 (штучне збільшення продажів у вихідні на 15–25%). Водночас розкид (висота «ящиків» та інтерквартильний розмах) у обох групах залишається порівнянним.

---

## Завдання 4. Діаграма розсіювання

Побудуємо діаграму розсіювання для виявлення зв'язку між продажами та середньою температурою.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df_kiosk["температура"], df_kiosk["продажі"], color="#d95f02", alpha=0.8)
ax.set_xlabel("Середня температура повітря, °C")
ax.set_ylabel("Кількість продажів за день, шт.")
ax.set_title("Продажі напоїв проти температури повітря")
plt.show()

**Результат виконання:**
*(Діаграма розсіювання виводиться у середовищі виконання)*

**Опис напрямку зв'язку:**

На графіку спостерігається відсутність вираженого сильного тренду або навіть слабкий зворотний/випадковий зв'язок (або відсутність кореляції), оскільки у нашому генераторі температура коливалася навколо стабільного літнього показника (32°C), а на продажі переважно впливав фактор дня тижня (будні/вихідні). Для реального кіоску гарячих/холодних напоїв у спекотний місяць (32°C) мінливість температури в межах ±1.5°C надто мала, щоб суттєво змінити поведінку покупців.

---

## Завдання 5. Навмисно спотворений графік

Продемонструємо ефект спотворення за допомогою штучного обрізання осі Y, порівнявши його з коректним графіком поруч.

In [ ]:
fig, (ax_correct, ax_distorted) = plt.subplots(1, 2, figsize=(12, 4))

categories = ["Будні", "Вихідні"]
medians = [weekday_sales.median(), weekend_sales.median()]

# 1. Коректна версія (вісь Y починається з 0)
ax_correct.bar(categories, medians, color=["#4daf4a", "#377eb8"], width=0.5)
ax_correct.set_ylim(0, 60)
ax_correct.set_ylabel("Медіана продажів, шт.")
ax_correct.set_title("Коректна вісь Y (від 0)")

# 2. Спотворена версія (штучно обрізана вісь Y від 35 до 55)
ax_distorted.bar(categories, medians, color=["#4daf4a", "#377eb8"], width=0.5)
ax_distorted.set_ylim(35, 55)
ax_distorted.set_ylabel("Медіана продажів, шт.")
ax_distorted.set_title("Спотворена вісь Y (обрізана від 35)")

plt.tight_layout()
plt.show()

**Результат виконання:**
*(Порівняння двох графіків виводиться у середовищі виконання)*

**Пояснення спотворення:**

У спотвореному графіку застосовано ефект **обрізаної осі Y (Truncated Axis)**. Оскільки вісь починається з 35 замість 0, стовпчик вихідних днів (48) здається візуально майже втричі вищим за стовпчик будніх днів (40), хоча реальна різниця становить лише близько 20%. Це маніпулює сприйняттям глядача, штучно перебільшуючи значущість різниці між категоріями.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, чому обрізана вісь Y на стовпчиковій діаграмі перебільшує різницу між категоріями, хоча підписані числа лишаються правильними?**
Висота стовпчиків на діаграмі сприймається зоровою системою людини пропорційно до площі або лінійного розміру. Якщо вісь Y не починається з нуля, співвідношення висот стовпчиків відображає відношення їхніх відхилень від нижньої межі обрізаної осі, а не реальне відношення самих числових величин, що створює хибне уявлення про масштаб явища.
2. **Чи можете навести приклад, коли лінійний графік (з'єднання точок лінією) буде методологічною помилковою для категоріальних даних, і пояснити чому — з посиланням на шкали вимірювання з Лекції 4?**
Лінійний графік призначений для неперервних або часових даних (інтервальна або відносна шкала), де з'єднання точок лінією імпліцитно означає наявність проміжних значень між ними. Якщо застосувати його до номінальних даних (наприклад, порівняння міст: Київ — Львів — Одеса), лінія між ними не має жодного математичного чи фізичного змісту, оскільки категорії не утворюють неперервної шкали.
3. **Чи розумієте чому підписи осей, одиниці вимірювання і заголовок — не оздоблення графіка, а частина твердження, яке він робить?**
Графік без підписів осей, одиниць та заголовка є амбіційним і неінформативним набором геометричних фігур. Заголовок визначає контекст гіпотези, а осі з одиницями задають метрики та масштаб, перетворюючи візуалізацію на чітке і перевірюване кількісне твердження.
4. **Чи можете пояснити, що таке chartjunk (наприклад, 3D-ефект на звичайній стовпчиковій діаграмі) і чому він шкодить, а не прикрашає сприйняття графіка?**
`Chartjunk` — це надлишкові візуальні елементи (зайві сітки, тіні, 3D-об'єми, патерни), які не несуть жодної корисної інформації про дані. 3D-ефекти на стовпчиках часто спотворюють реальну висоту значень через перспективу, ускладнюють зчитування даних і відволікають увагу глядача від суті аналізу.

---

## Висновок

Практичну роботу успішно виконано. За допомогою об'єктного інтерфейсу Matplotlib (`Figure`, `Axes`) згенеровано синтетичний набір даних продажів кіоску в Києві та побудовано гістограму, boxplot і діаграму розсіювання з дотриманням стандартів коректної візуалізації. Навмисно побудовано спотворений графік (з обрізаною віссю Y) для аналізу маніпуляцій у візуалізації даних.